In [23]:
import numpy as np
from scipy import fftpack, integrate
from math import pi
import csv, time
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [24]:
S0  = 100.0
r   = 0.0
EPS = 1e-12

STRIKES_PCT = np.array([0.80, 0.85, 0.90, 0.95, 0.97, 0.99,
                         1.00, 1.01, 1.02, 1.05, 1.10, 1.15, 1.20])
MATURITIES  = np.array([0.1, 0.25, 0.5, 1.0])

TARGET_FILAS          = 200_000
PUNTOS_POR_SUPERFICIE = len(STRIKES_PCT) * len(MATURITIES)
N_SUPERFICIES         = int(np.ceil(TARGET_FILAS / PUNTOS_POR_SUPERFICIE))
OUTFILE               = "dataset_heston_200k.csv"

SEED      = 42
TEST_SIZE = 0.15
VAL_SIZE  = 0.15
PARAMS_COLS = ['kappa', 'theta', 'sigma', 'rho', 'v0']

In [25]:
def caracteristica_heston(phi, params, S0_local, r_local, T, Pnum=2):
    kappa, theta, sigma, rho, v0 = params
    phi    = np.array(phi, dtype=complex)
    i      = 1j
    u      = 0.5 if Pnum == 1 else -0.5
    b      = (kappa - rho * sigma) if Pnum == 1 else kappa
    a      = kappa * theta
    sigma2 = sigma * sigma                           # una multiplicación en vez de **2
    sigma2 = sigma2 if sigma2 > EPS else EPS         # escalar, sin numpy overhead

    alpha = -0.5 * (phi * phi) + i * u * phi        # phi*phi en vez de phi**2
    beta  = b - rho * sigma * i * phi
    d     = np.sqrt(beta * beta - 2.0 * sigma2 * alpha)
    g     = (beta - d) / (beta + d)

    exp_dt         = np.exp(-d * T)
    one_minus_gexp = 1.0 - g * exp_dt
    one_minus_g    = 1.0 - g

    one_minus_gexp = np.where(np.abs(one_minus_gexp) < EPS, EPS, one_minus_gexp)
    one_minus_g    = np.where(np.abs(one_minus_g)    < EPS, EPS, one_minus_g)

    log_term = np.log(one_minus_gexp / one_minus_g)
    C = (i * phi * r_local * T) + (a / sigma2) * ((beta - d) * T - 2.0 * log_term)
    D = ((beta - d) / sigma2) * (1.0 - exp_dt) / one_minus_gexp

    return np.exp(C + D * v0 + i * phi * np.log(S0_local))

In [26]:
def _integrando(phi, params, S0_local, r_local, T, K, Pnum):
    cf  = caracteristica_heston(phi, params, S0_local, r_local, T, Pnum)
    val = np.exp(-1j * phi * np.log(K)) * cf / (1j * phi)
    return np.real(val)


def heston_precio_cerrado(params, S0_local, r_local, T, K, tol=1e-6, limit=200):
    I2, _ = integrate.quad(_integrando, 0, np.inf,
                            args=(params, S0_local, r_local, T, K, 2),
                            limit=limit, epsabs=tol, epsrel=tol)
    P2 = 0.5 + I2 / pi

    I1, _ = integrate.quad(_integrando, 0, np.inf,
                            args=(params, S0_local, r_local, T, K, 1),
                            limit=limit, epsabs=tol, epsrel=tol)
    P1 = 0.5 + I1 / pi

    return max(0.0, S0_local * P1 - K * np.exp(-r_local * T) * P2)


def verificar_formula_cerrada(params, T=0.5, K=100.0):
    precio_cerrado = heston_precio_cerrado(params, S0, r, T, K)
    K_fft, P_fft   = carr_madan_fft(params, S0, r, T)
    precio_cm      = float(np.interp(K, K_fft, P_fft))

    print(f"\n  Heston 1993 (integración directa) : {precio_cerrado:.6f} €")
    print(f"  Carr-Madan FFT                    : {precio_cm:.6f} €")
    print(f"  Diferencia                        : {abs(precio_cerrado - precio_cm):.2e} €")
    return precio_cerrado, precio_cm

In [27]:
def carr_madan_fft(params, S0_local, r_local, T, alpha=1.5, N=4096, eta=0.2):
    i     = 1j
    lambd = 2.0 * pi / (N * eta)
    b     = N * lambd / 2.0
    v     = np.arange(N, dtype=float) * eta          # dtype explícito, evita int array

    # Precalcular exp(-r*T) y -i*b*v una sola vez
    disc      = np.exp(-r_local * T)
    exp_ibv   = np.exp(-i * b * v)
    phi_shift = v - (alpha + 1.0) * i

    char_vals = caracteristica_heston(phi_shift, params, S0_local, r_local, T, Pnum=2)
    numerator = disc * char_vals * exp_ibv
    denom     = (alpha + i * v) * (alpha + i * v + 1.0)

    # fft sobre array ya real-multiplicado; .real al final de todo
    raw       = fftpack.fft((numerator / denom) * eta)

    k       = -b + np.arange(N, dtype=float) * lambd
    K_grid  = np.exp(k)
    precios = np.exp(-alpha * k) / pi * raw.real     # .real solo una vez, al final

    return K_grid, precios

In [28]:
def generar_paramsets_validos(n_needed, seed=SEED):
    rng        = np.random.default_rng(seed)
    valid_sets = []
    intentos   = 0
    while len(valid_sets) < n_needed:
        intentos += 1
        k   = rng.uniform(0.5, 5.0)
        t   = rng.uniform(0.01, 0.15)
        s   = rng.uniform(0.01, 0.6)
        rho = rng.uniform(-0.9, -0.1)
        v0  = rng.uniform(0.01, 0.2)
        if 2 * k * t > s**2:
            valid_sets.append([k, t, s, rho, v0])

    tasa = len(valid_sets) / intentos * 100
    print(f"  Parámetros válidos: {n_needed} de {intentos} intentos ({tasa:.1f}% aceptación)")
    return np.array(valid_sets)


def crear_dataset(n_superficies, outfile=OUTFILE):
    print(f"\n  Superficies : {n_superficies:,}  |  Puntos/sup: {PUNTOS_POR_SUPERFICIE}  |  Objetivo: {n_superficies * PUNTOS_POR_SUPERFICIE:,} filas")

    paramsets   = generar_paramsets_validos(n_superficies)
    strike_grid = S0 * STRIKES_PCT
    header      = ["kappa", "theta", "sigma", "rho", "v0", "T", "K", "price", "price_noisy"]
    rng         = np.random.default_rng(SEED)
    written     = 0
    errores     = 0
    t_start     = time.time()

    with open(outfile, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)

        for idx, params in enumerate(paramsets):
            if (idx + 1) % 500 == 0:
                elapsed = time.time() - t_start
                rate    = written / elapsed if elapsed > 0 else 1
                eta_s   = (TARGET_FILAS - written) / rate
                print(f"  [{idx+1:>5}/{n_superficies}]  {written:>8,} filas  "
                      f"({rate:.0f} filas/s)  ETA: {eta_s:.0f}s")

            for T in MATURITIES:
                try:
                    K_fft, P_fft   = carr_madan_fft(params, S0, r, T)
                    precios_interp = np.interp(strike_grid, K_fft, P_fft)
                    for j, K_val in enumerate(strike_grid):
                        base  = max(0.0, float(precios_interp[j]))
                        noisy = max(0.0, base * (1.0 + rng.normal(0.0, 0.01)))
                        writer.writerow([*params, float(T), float(K_val), base, noisy])
                        written += 1
                except Exception:
                    errores += 1
                    continue

    print(f"\n  Archivo   : {outfile}")
    print(f"  Filas     : {written:,}")
    print(f"  Errores   : {errores}")
    print(f"  Tiempo    : {(time.time()-t_start):.1f} s")
    return pd.read_csv(outfile)

In [29]:
def validar_dataset(df):
    feller      = (2 * df['kappa'] * df['theta'] > df['sigma']**2).all()
    intrinseco  = (S0 - df['K']).clip(lower=0)
    violaciones = (df['price'] < intrinseco - 1e-5).sum()
    precios_neg = (df['price'] < 0).sum()

    print(f"\n  Filas totales    : {len(df):,}")
    print(f"  Paramsets únicos : {df[PARAMS_COLS].drop_duplicates().shape[0]:,}")
    print(f"  Feller cumplido  : {feller}  {'✓' if feller else '✗ REVISAR'}")
    print(f"  Precios negativos: {precios_neg}  {'✓' if precios_neg == 0 else '✗ REVISAR'}")
    print(f"  Violac. arbitraje: {violaciones}  {'✓' if violaciones == 0 else '✗ REVISAR'}")

    p0  = df.iloc[0][PARAMS_COLS].values
    sub = df[(df['kappa'] == p0[0]) & (df['theta'] == p0[1])]

    fig = plt.figure(figsize=(10, 6))
    ax  = fig.add_subplot(111, projection='3d')
    ax.plot_trisurf(sub['K'], sub['T'], sub['price'], cmap='viridis', alpha=0.85)
    ax.set_title(f"Superficie de Precios Heston (muestra)\n"
                 f"κ={p0[0]:.2f}, θ={p0[1]:.4f}, σ={p0[2]:.2f}, ρ={p0[3]:.2f}, v₀={p0[4]:.4f}")
    ax.set_xlabel("Strike K (€)")
    ax.set_ylabel("Vencimiento T (años)")
    ax.set_zlabel("Precio call (€)")
    plt.tight_layout()
    plt.savefig("superficie_heston.png", dpi=130)
    plt.show()

In [30]:
def preparar_datos(df):
    X = pd.DataFrame()
    X['log_moneyness'] = np.log(df['K'] / S0)
    X['sqrt_T']        = np.sqrt(df['T'].clip(lower=1e-12))
    X['kappa']         = df['kappa']
    X['log_theta']     = np.log(df['theta'].clip(lower=1e-8))
    X['sigma']         = df['sigma']
    X['rho']           = df['rho']
    X['log_v0']        = np.log(df['v0'].clip(lower=1e-8))

    y = (df['price'] / S0).values

    paramsets = df[PARAMS_COLS].drop_duplicates().reset_index(drop=True)
    rng       = np.random.default_rng(SEED)
    perm      = rng.permutation(len(paramsets))
    n_test    = int(len(paramsets) * TEST_SIZE)
    n_val     = int(len(paramsets) * VAL_SIZE)

    ps_test  = paramsets.iloc[perm[:n_test]]
    ps_val   = paramsets.iloc[perm[n_test:n_test + n_val]]
    ps_train = paramsets.iloc[perm[n_test + n_val:]]

    # ── Reemplaza merge+indicator por isin sobre tuplas (mucho más rápido) ──
    df_tuples  = list(zip(*[df[c] for c in PARAMS_COLS]))

    def make_mask(ps):
        ps_set = set(map(tuple, ps.values))
        return np.array([t in ps_set for t in df_tuples], dtype=bool)

    m_train = make_mask(ps_train)
    m_val   = make_mask(ps_val)
    m_test  = make_mask(ps_test)

    X_vals = X.values
    X_train_raw, y_train = X_vals[m_train], y[m_train]
    X_val_raw,   y_val   = X_vals[m_val],   y[m_val]
    X_test_raw,  y_test  = X_vals[m_test],  y[m_test]

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val   = scaler.transform(X_val_raw)
    X_test  = scaler.transform(X_test_raw)

    joblib.dump(scaler, 'escalador_heston_nn.pkl')

    print(f"\n  Train : {X_train.shape[0]:>8,} filas  ({len(ps_train)} superficies)")
    print(f"  Val   : {X_val.shape[0]:>8,} filas  ({len(ps_val)} superficies)")
    print(f"  Test  : {X_test.shape[0]:>8,} filas  ({len(ps_test)} superficies)")

    return X_train, X_val, X_test, y_train, y_val, y_test, scaler

In [31]:
df_final = crear_dataset(N_SUPERFICIES, outfile=OUTFILE)


  Superficies : 3,847  |  Puntos/sup: 52  |  Objetivo: 200,044 filas
  Parámetros válidos: 3847 de 4577 intentos (84.1% aceptación)
  [  500/3847]    25,948 filas  (6206 filas/s)  ETA: 28s
  [ 1000/3847]    51,948 filas  (6234 filas/s)  ETA: 24s
  [ 1500/3847]    77,948 filas  (6233 filas/s)  ETA: 20s
  [ 2000/3847]   103,948 filas  (6237 filas/s)  ETA: 15s
  [ 2500/3847]   129,948 filas  (6243 filas/s)  ETA: 11s
  [ 3000/3847]   155,948 filas  (6233 filas/s)  ETA: 7s
  [ 3500/3847]   181,948 filas  (6224 filas/s)  ETA: 3s

  Archivo   : dataset_heston_200k.csv
  Filas     : 200,044
  Errores   : 0
  Tiempo    : 32.2 s


In [32]:
X_train, X_val, X_test, y_train, y_val, y_test, scaler = preparar_datos(df_final)


  Train :  140,036 filas  (2693 superficies)
  Val   :   30,004 filas  (577 superficies)
  Test  :   30,004 filas  (577 superficies)


In [33]:
INPUT_DIM = X_train.shape[1]

In [34]:
X_train.shape

(140036, 7)

In [35]:
def build_model(hp):
    """
    Construye un MLP cuyos hiperparámetros son decididos por `hp`.

    Parámetros optimizables:
      hp.Int      → entero en un rango [min, max] con un paso (step)
      hp.Float    → flotante en un rango [min, max] (lineal o log)
      hp.Choice   → elige entre una lista de valores posibles
      hp.Boolean  → True / False
    """

    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(INPUT_DIM,), name="entrada"))

    # --- Número de capas ocultas ---
    n_layers = hp.Int(
        name="n_layers",
        min_value=1,
        max_value=5,
        step=1
    )

    for i in range(n_layers):

        # --- Unidades de la capa i ---
        units = hp.Int(
            name=f"units_layer_{i}",
            min_value=32,
            max_value=512,
            step=32
        )

        # --- Función de activación ---
        activation = hp.Choice(
            name=f"activation_layer_{i}",
            values=["relu", "tanh", "selu", "elu"]
        )

        model.add(keras.layers.Dense(units=units, activation=activation))

        # --- Batch Normalization ---
        use_bn = hp.Boolean(name=f"batch_norm_layer_{i}", default=False)
        if use_bn:
            model.add(keras.layers.BatchNormalization())

        # --- Dropout ---
        dropout_rate = hp.Float(
            name=f"dropout_layer_{i}",
            min_value=0.0,
            max_value=0.5,
            step=0.05
        )
        if dropout_rate > 0:
            model.add(keras.layers.Dropout(rate=dropout_rate))

    # Capa de salida: 1 neurona, sin activación (regresión)
    model.add(keras.layers.Dense(1))

    # --- Learning rate ---
    learning_rate = hp.Float(
        name="learning_rate",
        min_value=1e-4,
        max_value=1e-2,
        sampling="log",   # búsqueda en escala logarítmica
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [37]:
tuner_rs = kt.RandomSearch(
    hypermodel=build_model,
    objective="val_mae",            # métrica a minimizar
    max_trials=20,                  # cuántas configuraciones aleatorias probar
    directory="kt_results",         # carpeta donde se guardan los resultados
    project_name="mlp_regression",  # subcarpeta del proyecto
    overwrite=True                  # sobreescribir resultados anteriores
)

# Resumen del espacio de búsqueda
tuner_rs.search_space_summary()

Search space summary
Default search space size: 6
n_layers (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 5, 'step': 1, 'sampling': 'linear'}
units_layer_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
activation_layer_0 (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'tanh', 'selu', 'elu'], 'ordered': False}
batch_norm_layer_0 (Boolean)
{'default': False, 'conditions': []}
dropout_layer_0 (Float)
{'default': 0.0, 'conditions': [], 'min_value': 0.0, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


In [38]:
callbacks = [
    # Detiene el entrenamiento si val_loss no mejora en N épocas
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce la tasa de aprendizaje si el modelo deja de aprender
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    ),
    # Guarda el mejor modelo de vez en cuando por si acaso hay algún problema
    keras.callbacks.ModelCheckpoint(
        filepath='mejor_modelo_heston.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

In [40]:
tuner_rs.search(
    X_train, y_train,
    validation_split=0.15,   # fracción de X_train usada como validación
    epochs=20,               # épocas por trial
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

Trial 20 Complete [00h 01m 23s]
val_mae: 0.005996927618980408

Best val_mae So Far: 0.0016050304984673858
Total elapsed time: 00h 25m 06s


In [41]:
best_hps_rs = tuner_rs.get_best_hyperparameters(num_trials=1)[0]
print("\n=== Mejores hiperparámetros ===")
print(f"  Capas ocultas : {best_hps_rs.get('n_layers')}")
print(f"  Learning rate : {best_hps_rs.get('learning_rate'):.5f}")
for i in range(best_hps_rs.get("n_layers")):
    print(f"  Capa {i}: units={best_hps_rs.get(f'units_layer_{i}')}, "
          f"act={best_hps_rs.get(f'activation_layer_{i}')}, "
          f"dropout={best_hps_rs.get(f'dropout_layer_{i}'):.2f}, ")


=== Mejores hiperparámetros ===
  Capas ocultas : 3
  Learning rate : 0.00580
  Capa 0: units=32, act=tanh, dropout=0.00, 
  Capa 1: units=32, act=tanh, dropout=0.00, 
  Capa 2: units=32, act=tanh, dropout=0.00, 


In [42]:
best_model_rs = tuner_rs.get_best_models(num_models=1)[0]

history_rs = best_model_rs.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1860/1860 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 5.8400e-05 - mae: 0.0060 - val_loss: 1.7420e-05 - val_mae: 0.0033 - learning_rate: 0.0058
Epoch 2/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 4.7608e-05 - mae: 0.0054 - val_loss: 2.3351e-05 - val_mae: 0.0037 - learning_rate: 0.0058
Epoch 3/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 3.5650e-05 - mae: 0.0047 - val_loss: 2.3987e-05 - val_mae: 0.0040 - learning_rate: 0.0058
Epoch 4/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 3.3052e-05 - mae: 0.0045 - val_loss: 9.2556e-06 - val_mae: 0.0021 - learning_rate: 0.0058
Epoch 5/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 3.0948e-05 - mae: 0.0044 - val_loss: 4.0375e-05 - val_mae: 0.0056 - learning_rate: 0.0058
Epoch 6/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 3.0574e-05 - mae: 0.0044 - val_loss: 3.4808e-05 - val_mae: 0.0048 - learning_rate: 0.0058
Epoch 7/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 2.9458e-05 - mae: 0.0043 - val

In [53]:
test_loss_rs, test_mae_rs = best_model_rs.evaluate(X_test, y_test, verbose=0)
print(f"\n=== Evaluación en test ===")
print(f"  MSE : {test_loss_rs:.7f}")
print(f"  MAE : {test_mae_rs:.7f}")

# Predicciones
y_pred_rs = best_model_rs.predict(X_test).ravel() # .ravel más eficiente que flatten

print(f"\nPrimeras 5 predicciones vs valores reales:")
for pred, real in zip(y_pred_rs[:5], y_test[:5]):
    print(f"  pred={pred:.3f}  real={real:.3f}")


=== Evaluación en test ===
  MSE : 0.0000010
  MAE : 0.0007523
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

Primeras 5 predicciones vs valores reales:
  pred=0.214  real=0.214
  pred=0.165  real=0.166
  pred=0.123  real=0.122
  pred=0.085  real=0.085
  pred=0.073  real=0.072


In [44]:
best_model_rs.save("best_mlp_model_rs.keras")
print("\nModelo guardado en best_mlp_model_rs.keras")


Modelo guardado en best_mlp_model_rs.keras


In [45]:
tuner_hb = kt.Hyperband(            # Hyperband: elimina configuraciones malas pronto, por lo que no prueba todas y es bastante optimizado
    hypermodel=build_model,
    objective="val_mae",            # métrica a minimizar
    max_epochs=20,                  # épocas máximas por trial en Hyperband
    factor=3,                       # factor de reducción de Hyperband
    directory="kt_results",         # carpeta donde se guardan los resultados
    project_name="mlp_regression",  # subcarpeta del proyecto
    overwrite=True                  # sobreescribir resultados anteriores
)

# Resumen del espacio de búsqueda
tuner_hb.search_space_summary()

Search space summary
Default search space size: 6
n_layers (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 5, 'step': 1, 'sampling': 'linear'}
units_layer_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
activation_layer_0 (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'tanh', 'selu', 'elu'], 'ordered': False}
batch_norm_layer_0 (Boolean)
{'default': False, 'conditions': []}
dropout_layer_0 (Float)
{'default': 0.0, 'conditions': [], 'min_value': 0.0, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


In [46]:
callbacks = [
    # Detiene el entrenamiento si val_loss no mejora en N épocas
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce la tasa de aprendizaje si el modelo deja de aprender
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    ),
    # Guarda el mejor modelo de vez en cuando por si acaso hay algún problema
    keras.callbacks.ModelCheckpoint(
        filepath='mejor_modelo_heston.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

In [47]:
tuner_hb.search(
    X_train, y_train,
    validation_split=0.15,   # fracción de X_train usada como validación
    epochs=20,               # épocas por trial (Hyperband las gestiona internamente)
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

Trial 30 Complete [00h 01m 06s]
val_mae: 0.01184450276196003

Best val_mae So Far: 0.0015086766798049212
Total elapsed time: 00h 16m 41s


In [61]:
best_hps_hb = tuner_hb.get_best_hyperparameters(num_trials=1)[0]
print("\n=== Mejores hiperparámetros ===")
print(f"  Capas ocultas : {best_hps_hb.get('n_layers')}")
print(f"  Learning rate : {best_hps_hb.get('learning_rate'):.4f}")
for i in range(best_hps_hb.get("n_layers")):
    print(f"  Capa {i}: units={best_hps_hb.get(f'units_layer_{i}')}, "
          f"act={best_hps_hb.get(f'activation_layer_{i}')}, "
          f"dropout={best_hps_hb.get(f'dropout_layer_{i}'):.2f}, ")


=== Mejores hiperparámetros ===
  Capas ocultas : 5
  Learning rate : 0.0005
  Capa 0: units=416, act=tanh, dropout=0.05, 
  Capa 1: units=128, act=tanh, dropout=0.35, 
  Capa 2: units=32, act=relu, dropout=0.00, 
  Capa 3: units=32, act=relu, dropout=0.00, 
  Capa 4: units=32, act=relu, dropout=0.00, 


In [49]:
best_model_hb = tuner_hb.get_best_models(num_models=1)[0]

history_hb = best_model_hb.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 26s 9ms/step - loss: 4.1468e-05 - mae: 0.0049 - val_loss: 1.3846e-05 - val_mae: 0.0031 - learning_rate: 4.9113e-04
Epoch 2/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 3.3805e-05 - mae: 0.0044 - val_loss: 1.1124e-05 - val_mae: 0.0027 - learning_rate: 4.9113e-04
Epoch 3/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 3.0373e-05 - mae: 0.0042 - val_loss: 6.5133e-06 - val_mae: 0.0021 - learning_rate: 4.9113e-04
Epoch 4/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 2.6556e-05 - mae: 0.0039 - val_loss: 5.7018e-06 - val_mae: 0.0018 - learning_rate: 4.9113e-04
Epoch 5/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 2.4370e-05 - mae: 0.0038 - val_loss: 5.5091e-06 - val_mae: 0.0019 - learning_rate: 4.9113e-04
Epoch 6/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 2.2627e-05 - mae: 0.0036 - val_loss: 6.1785e-06 - val_mae: 0.0020 - learning_rate: 4.9113e-04
Epoch 7/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - 

In [63]:
test_loss_hb, test_mae_hb = best_model_hb.evaluate(X_test, y_test, verbose=0)
print(f"\n=== Evaluación en test ===")
print(f"  MSE : {test_loss_hb:.7f}")
print(f"  MAE : {test_mae_hb:.7f}")

# Predicciones
y_pred_hb = best_model_hb.predict(X_test).ravel() # .ravel más eficiente que flatten

print(f"\nPrimeras 5 predicciones vs valores reales:")
for pred, real in zip(y_pred_hb[:5], y_test[:5]):
    print(f"  pred={pred:.3f}  real={real:.3f}")


=== Evaluación en test ===
  MSE : 0.0000015
  MAE : 0.0009140
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

Primeras 5 predicciones vs valores reales:
  pred=0.214  real=0.214
  pred=0.165  real=0.166
  pred=0.122  real=0.122
  pred=0.085  real=0.085
  pred=0.072  real=0.072


In [51]:
best_model_hb.save("best_mlp_model_hb.keras")
print("\nModelo guardado en best_mlp_model_hb.keras")


Modelo guardado en best_mlp_model_hb.keras


In [64]:
tuner_bo = kt.BayesianOptimization(
    hypermodel=build_model,
    objective="val_mae",            # métrica a minimizar
    max_trials=20,                  # cuántas configuraciones aleatorias probar
    num_initial_points=5,           # trials aleatorios antes de empezar a modelar (≥2)
    seed=33,
    directory="kt_results",         # carpeta donde se guardan los resultados
    project_name="mlp_regression",  # subcarpeta del proyecto
    overwrite=True                  # sobreescribir resultados anteriores
)

# Resumen del espacio de búsqueda
tuner_bo.search_space_summary()

Search space summary
Default search space size: 6
n_layers (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 5, 'step': 1, 'sampling': 'linear'}
units_layer_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
activation_layer_0 (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'tanh', 'selu', 'elu'], 'ordered': False}
batch_norm_layer_0 (Boolean)
{'default': False, 'conditions': []}
dropout_layer_0 (Float)
{'default': 0.0, 'conditions': [], 'min_value': 0.0, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


In [65]:
callbacks = [
    # Detiene el entrenamiento si val_loss no mejora en N épocas
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce la tasa de aprendizaje si el modelo deja de aprender
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    ),
    # Guarda el mejor modelo de vez en cuando por si acaso hay algún problema
    keras.callbacks.ModelCheckpoint(
        filepath='mejor_modelo_heston.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

In [66]:
tuner_bo.search(
    X_train, y_train,
    validation_split=0.15,   # fracción de X_train usada como validación
    epochs=20,               # épocas por trial
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

Trial 20 Complete [00h 00m 48s]
val_mae: 0.004135972820222378

Best val_mae So Far: 0.00036338824429549277
Total elapsed time: 00h 20m 02s


In [67]:
best_hps_bo = tuner_bo.get_best_hyperparameters(num_trials=1)[0]
print("\n=== Mejores hiperparámetros ===")
print(f"  Capas ocultas : {best_hps_bo.get('n_layers')}")
print(f"  Learning rate : {best_hps_bo.get('learning_rate'):.5f}")
for i in range(best_hps_bo.get("n_layers")):
    print(f"  Capa {i}: units={best_hps_bo.get(f'units_layer_{i}')}, "
          f"act={best_hps_bo.get(f'activation_layer_{i}')}, "
          f"dropout={best_hps_bo.get(f'dropout_layer_{i}'):.2f}, ")


=== Mejores hiperparámetros ===
  Capas ocultas : 1
  Learning rate : 0.00132
  Capa 0: units=512, act=relu, dropout=0.00, 


In [68]:
best_model_bo = tuner_bo.get_best_models(num_models=1)[0]

history_bo = best_model_bo.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1860/1860 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 4.4934e-06 - mae: 0.0015 - val_loss: 2.5747e-06 - val_mae: 0.0012 - learning_rate: 0.0013
Epoch 2/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 2.3278e-06 - mae: 0.0011 - val_loss: 1.3118e-06 - val_mae: 9.0310e-04 - learning_rate: 0.0013
Epoch 3/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 1.7234e-06 - mae: 9.7438e-04 - val_loss: 1.6804e-06 - val_mae: 9.7654e-04 - learning_rate: 0.0013
Epoch 4/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 1.4260e-06 - mae: 8.8726e-04 - val_loss: 1.9536e-06 - val_mae: 0.0011 - learning_rate: 0.0013
Epoch 5/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 1.4113e-06 - mae: 8.8312e-04 - val_loss: 9.6256e-07 - val_mae: 7.3057e-04 - learning_rate: 0.0013
Epoch 6/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 1.1745e-06 - mae: 8.0645e-04 - val_loss: 5.8968e-07 - val_mae: 5.6048e-04 - learning_rate: 0.0013
Epoch 7/100
1860/1860 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss:

In [69]:
test_loss_bo, test_mae_bo = best_model_bo.evaluate(X_test, y_test, verbose=0)
print(f"\n=== Evaluación en test ===")
print(f"  MSE : {test_loss_bo:.7f}")
print(f"  MAE : {test_mae_bo:.7f}")

# Predicciones
y_pred_bo = best_model_bo.predict(X_test).ravel() # .ravel más eficiente que flatten

print(f"\nPrimeras 5 predicciones vs valores reales:")
for pred, real in zip(y_pred_bo[:5], y_test[:5]):
    print(f"  pred={pred:.3f}  real={real:.3f}")


=== Evaluación en test ===
  MSE : 0.0000001
  MAE : 0.0002626
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

Primeras 5 predicciones vs valores reales:
  pred=0.213  real=0.214
  pred=0.166  real=0.166
  pred=0.122  real=0.122
  pred=0.085  real=0.085
  pred=0.072  real=0.072


In [70]:
best_model_bo.save("best_mlp_model_bo.keras")
print("\nModelo guardado en best_mlp_model_bo.keras")


Modelo guardado en best_mlp_model_bo.keras
